# Privacy Meter Demo

This notebook is an interactive demonstration of running Privacy Meter to audit privacy defined by **MIA**, which is the default auditing methodology. For a detailed explanation on MIA and how to run from bash instead, please refer to the [documentation](documentation/mia.md)


## Setting up the Colab environment

If you are running it offline, you can skip to "Importing"

In [1]:
# Clone the github repo
#!git clone https://github.com/privacytrustlab/ml_privacy_meter.git

# Update the Colab environment
!pip install datasets==2.21.0 transformers==4.44.2 torch==2.4.1 torchvision==0.19.1 torchaudio


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Change the directory to the cloned repo
import sys
#sys.path.append('/content/ml_privacy_meter')

#%cd ml_privacy_meter

## Importing

In [3]:
import math
import time

import numpy as np
import torch
import yaml
from torch.utils.data import Subset

from audit import get_average_audit_results, audit_models, sample_auditing_dataset
from get_signals import get_model_signals
from models.utils import load_models, train_models, split_dataset_for_training
from util import (
    check_configs,
    setup_log,
    initialize_seeds,
    create_directories,
    load_dataset,
)

# Enable benchmark mode in cudnn to improve performance when input sizes are consistent
torch.backends.cudnn.benchmark = True

## Load config

In [4]:
configs = "configs/config.yaml"
with open(configs, "rb") as f:
        configs = yaml.load(f, Loader=yaml.Loader)

# Validate configurations
check_configs(configs)

## Setting up

In [5]:
# Validate configurations
check_configs(configs)

# Initialize seeds for reproducibility
initialize_seeds(configs["run"]["random_seed"])

# Create necessary directories
log_dir = configs["run"]["log_dir"]
directories = {
    "log_dir": log_dir,
    "report_dir": f"{log_dir}/report",
    "signal_dir": f"{log_dir}/signals",
    "data_dir": configs["data"]["data_dir"],
}
create_directories(directories)

# Set up logger
logger = setup_log(
    directories["report_dir"], "time_analysis", configs["run"]["time_log"]
)

start_time = time.time()

## Load dataset

In [6]:
baseline_time = time.time()
dataset, population = load_dataset(configs, directories["data_dir"], logger)
logger.info("Loading dataset took %0.5f seconds", time.time() - baseline_time)

2026-07-21 18:52:00,116 INFO     Data loaded from data/cifar10.pkl
2026-07-21 18:52:00,141 INFO     Population data loaded from data/cifar10_population.pkl
2026-07-21 18:52:00,141 INFO     The whole dataset size: 50000
2026-07-21 18:52:00,142 INFO     Loading dataset took 0.15900 seconds


## Load or train models

In [7]:
# Define experiment parameters
num_experiments = configs["run"]["num_experiments"]
num_reference_models = configs["audit"]["num_ref_models"]
num_model_pairs = max(math.ceil(num_experiments / 2.0), num_reference_models + 1)

# Load or train models
baseline_time = time.time()
models_list, memberships = load_models(
    log_dir, dataset, num_model_pairs * 2, configs, logger
)
if models_list is None:
    # Split dataset for training two models per pair
    data_splits, memberships = split_dataset_for_training(
        len(dataset), num_model_pairs
    )
    models_list = train_models(
        log_dir, dataset, data_splits, memberships, configs, logger
    )
logger.info(
    "Model loading/training took %0.1f seconds", time.time() - baseline_time
)


2026-07-21 18:52:00,160 INFO     Training 4 models
2026-07-21 18:52:00,163 INFO     --------------------------------------------------
2026-07-21 18:52:00,164 INFO     Training model 0: Train size 25000, Test size 25000


Using optimizer: SGD | Learning Rate: 0.1 | Weight Decay: 0
Epoch [1/100] | Train Loss: 2.2984 | Train Acc: 0.1201
Test Loss: 2.2163 | Test Acc: 0.1996
Epoch 1 took 49.84 seconds
Epoch [2/100] | Train Loss: 2.0591 | Train Acc: 0.2711
Test Loss: 1.9070 | Test Acc: 0.3024
Epoch 2 took 12.65 seconds
Epoch [3/100] | Train Loss: 1.8079 | Train Acc: 0.3500
Test Loss: 1.7261 | Test Acc: 0.3774
Epoch 3 took 12.71 seconds
Epoch [4/100] | Train Loss: 1.6541 | Train Acc: 0.3959
Test Loss: 1.6104 | Test Acc: 0.4191
Epoch 4 took 12.79 seconds
Epoch [5/100] | Train Loss: 1.5391 | Train Acc: 0.4392
Test Loss: 1.5663 | Test Acc: 0.4110
Epoch 5 took 12.87 seconds
Epoch [6/100] | Train Loss: 1.4468 | Train Acc: 0.4727
Test Loss: 1.4567 | Test Acc: 0.4685
Epoch 6 took 12.95 seconds
Epoch [7/100] | Train Loss: 1.3606 | Train Acc: 0.5067
Test Loss: 1.6694 | Test Acc: 0.4206
Epoch 7 took 13.03 seconds
Epoch [8/100] | Train Loss: 1.2890 | Train Acc: 0.5370
Test Loss: 1.5694 | Test Acc: 0.4300
Epoch 8 took 13

2026-07-21 19:15:24,053 INFO     Train accuracy 1.0, Train Loss 0.0030262979407965833
2026-07-21 19:15:24,056 INFO     Test accuracy 0.6354, Test Loss 1.4399348837988717
2026-07-21 19:15:24,088 INFO     Training model 0 took 1403.9248309135437 seconds
2026-07-21 19:15:24,136 INFO     --------------------------------------------------
2026-07-21 19:15:24,137 INFO     Training model 1: Train size 25000, Test size 25000


Using optimizer: SGD | Learning Rate: 0.1 | Weight Decay: 0
Epoch [1/100] | Train Loss: 2.2549 | Train Acc: 0.1548
Test Loss: 2.1138 | Test Acc: 0.2276
Epoch 1 took 45.26 seconds
Epoch [2/100] | Train Loss: 1.9565 | Train Acc: 0.2798
Test Loss: 1.8800 | Test Acc: 0.2914
Epoch 2 took 13.30 seconds
Epoch [3/100] | Train Loss: 1.7892 | Train Acc: 0.3444
Test Loss: 1.7854 | Test Acc: 0.3529
Epoch 3 took 13.48 seconds
Epoch [4/100] | Train Loss: 1.6840 | Train Acc: 0.3872
Test Loss: 1.6550 | Test Acc: 0.3896
Epoch 4 took 13.59 seconds
Epoch [5/100] | Train Loss: 1.5841 | Train Acc: 0.4250
Test Loss: 1.6547 | Test Acc: 0.3927
Epoch 5 took 13.64 seconds
Epoch [6/100] | Train Loss: 1.4926 | Train Acc: 0.4594
Test Loss: 1.5392 | Test Acc: 0.4236
Epoch 6 took 13.70 seconds
Epoch [7/100] | Train Loss: 1.4127 | Train Acc: 0.4889
Test Loss: 1.5970 | Test Acc: 0.4192
Epoch 7 took 13.73 seconds
Epoch [8/100] | Train Loss: 1.3345 | Train Acc: 0.5229
Test Loss: 1.4246 | Test Acc: 0.4945
Epoch 8 took 13

2026-07-21 19:39:07,052 INFO     Train accuracy 1.0, Train Loss 0.0025168380733313306
2026-07-21 19:39:07,052 INFO     Test accuracy 0.63624, Test Loss 1.4698389355017214
2026-07-21 19:39:07,067 INFO     Training model 1 took 1422.931762933731 seconds
2026-07-21 19:39:07,091 INFO     --------------------------------------------------
2026-07-21 19:39:07,091 INFO     Training model 2: Train size 25000, Test size 25000


Using optimizer: SGD | Learning Rate: 0.1 | Weight Decay: 0
Epoch [1/100] | Train Loss: 2.2583 | Train Acc: 0.1624
Test Loss: 2.1427 | Test Acc: 0.2175
Epoch 1 took 48.08 seconds
Epoch [2/100] | Train Loss: 1.9846 | Train Acc: 0.2845
Test Loss: 1.9033 | Test Acc: 0.3168
Epoch 2 took 13.32 seconds
Epoch [3/100] | Train Loss: 1.7717 | Train Acc: 0.3679
Test Loss: 1.7354 | Test Acc: 0.3712
Epoch 3 took 13.48 seconds
Epoch [4/100] | Train Loss: 1.6266 | Train Acc: 0.4180
Test Loss: 1.6048 | Test Acc: 0.4138
Epoch 4 took 13.60 seconds
Epoch [5/100] | Train Loss: 1.5190 | Train Acc: 0.4515
Test Loss: 1.5400 | Test Acc: 0.4400
Epoch 5 took 13.65 seconds
Epoch [6/100] | Train Loss: 1.4318 | Train Acc: 0.4825
Test Loss: 1.5631 | Test Acc: 0.4166
Epoch 6 took 13.75 seconds
Epoch [7/100] | Train Loss: 1.3556 | Train Acc: 0.5112
Test Loss: 1.5131 | Test Acc: 0.4420
Epoch 7 took 13.79 seconds
Epoch [8/100] | Train Loss: 1.2850 | Train Acc: 0.5371
Test Loss: 1.3571 | Test Acc: 0.5059
Epoch 8 took 13

2026-07-21 20:02:55,195 INFO     Train accuracy 1.0, Train Loss 0.003048959444752153
2026-07-21 20:02:55,196 INFO     Test accuracy 0.62592, Test Loss 1.5161160668548272
2026-07-21 20:02:55,214 INFO     Training model 2 took 1428.1222755908966 seconds
2026-07-21 20:02:55,241 INFO     --------------------------------------------------
2026-07-21 20:02:55,242 INFO     Training model 3: Train size 25000, Test size 25000


Using optimizer: SGD | Learning Rate: 0.1 | Weight Decay: 0
Epoch [1/100] | Train Loss: 2.2646 | Train Acc: 0.1318
Test Loss: 2.1036 | Test Acc: 0.2180
Epoch 1 took 45.69 seconds
Epoch [2/100] | Train Loss: 1.9908 | Train Acc: 0.2770
Test Loss: 1.8750 | Test Acc: 0.3261
Epoch 2 took 13.26 seconds
Epoch [3/100] | Train Loss: 1.8039 | Train Acc: 0.3478
Test Loss: 1.7462 | Test Acc: 0.3704
Epoch 3 took 13.45 seconds
Epoch [4/100] | Train Loss: 1.6709 | Train Acc: 0.3902
Test Loss: 1.6220 | Test Acc: 0.3985
Epoch 4 took 13.58 seconds
Epoch [5/100] | Train Loss: 1.5709 | Train Acc: 0.4216
Test Loss: 1.5521 | Test Acc: 0.4329
Epoch 5 took 13.61 seconds
Epoch [6/100] | Train Loss: 1.4858 | Train Acc: 0.4538
Test Loss: 1.4774 | Test Acc: 0.4600
Epoch 6 took 13.67 seconds
Epoch [7/100] | Train Loss: 1.4138 | Train Acc: 0.4866
Test Loss: 1.4981 | Test Acc: 0.4465
Epoch 7 took 13.71 seconds
Epoch [8/100] | Train Loss: 1.3440 | Train Acc: 0.5163
Test Loss: 1.3475 | Test Acc: 0.5150
Epoch 8 took 13

2026-07-21 20:26:36,826 INFO     Train accuracy 1.0, Train Loss 0.0035022786813693084
2026-07-21 20:26:36,826 INFO     Test accuracy 0.62604, Test Loss 1.5374278109900805
2026-07-21 20:26:36,841 INFO     Training model 3 took 1421.600385427475 seconds
2026-07-21 20:26:38,362 INFO     Model loading/training took 5678.2 seconds


## Prepare auditing dataset

In [8]:
auditing_dataset, auditing_membership = sample_auditing_dataset(
        configs, dataset, logger, memberships
    )

# Also downsample the population set size if specified in the config
population = Subset(
    population,
    np.random.choice(
        len(population),
        configs["audit"].get("population_size", len(population)),
        replace=False,
    ),
)

## Compute signals

In [9]:
baseline_time = time.time()
signals = get_model_signals(models_list, auditing_dataset, configs, logger)
population_signals = get_model_signals(
        models_list, population, configs, logger, is_population=True
    )
logger.info("Preparing signals took %0.5f seconds", time.time() - baseline_time)

2026-07-21 20:26:57,961 INFO     Computing signals for all models.
Computing softmax: 100%|██████████| 10/10 [00:15<00:00,  1.58s/it]
2026-07-21 20:28:32,182 INFO     Signals saved to disk.
2026-07-21 20:28:53,636 INFO     Computing signals for all models.
Computing softmax: 100%|██████████| 2/2 [00:03<00:00,  1.52s/it]
2026-07-21 20:29:06,376 INFO     Signals saved to disk.
2026-07-21 20:29:06,377 INFO     Preparing signals took 147.94122 seconds


## Audit

In [10]:
# Perform the privacy audit
baseline_time = time.time()
target_model_indices = list(range(num_experiments))
mia_score_list, membership_list = audit_models(
        f"{directories['report_dir']}/exp",
        target_model_indices,
        signals,
        population_signals,
        auditing_membership,
        num_reference_models,
        logger,
        configs,
    )

if len(target_model_indices) > 1:
    logger.info(
        "Auditing privacy risk took %0.1f seconds", time.time() - baseline_time
    )

# Get average audit results across all experiments
if len(target_model_indices) > 1:
    get_average_audit_results(
        directories["report_dir"], mia_score_list, membership_list, logger
    )

logger.info("Total runtime: %0.5f seconds", time.time() - start_time)

2026-07-21 20:29:06,395 INFO     Fine-tuning offline_a using paired model 1
2026-07-21 20:29:07,808 INFO     offline_a=0.00: AUC 0.8183
2026-07-21 20:29:09,009 INFO     offline_a=0.10: AUC 0.8137
2026-07-21 20:29:10,166 INFO     offline_a=0.20: AUC 0.8089
2026-07-21 20:29:11,278 INFO     offline_a=0.30: AUC 0.8038
2026-07-21 20:29:12,612 INFO     offline_a=0.40: AUC 0.7987
2026-07-21 20:29:13,871 INFO     offline_a=0.50: AUC 0.7928
2026-07-21 20:29:15,069 INFO     offline_a=0.60: AUC 0.7862
2026-07-21 20:29:16,283 INFO     offline_a=0.70: AUC 0.7789
2026-07-21 20:29:17,350 INFO     offline_a=0.80: AUC 0.7693
2026-07-21 20:29:18,592 INFO     offline_a=0.90: AUC 0.7562
2026-07-21 20:29:19,810 INFO     offline_a=1.00: AUC 0.7044
2026-07-21 20:29:19,812 INFO     The best offline_a is 0.0
2026-07-21 20:29:20,991 INFO     Target Model 0: AUC 0.8115, TPR@0.1%FPR of 0.0074, TPR@0.0%FPR of 0.0000
2026-07-21 20:29:24,006 INFO     Auditing the privacy risks of target model 0 costs 17.6 seconds
20

<Figure size 640x480 with 0 Axes>